In [3]:
from pathlib import Path

pdf_path = Path("pdf/IS4151_2015.pdf")

In [4]:
import subprocess
import sys
import xml.etree.ElementTree as ET
import tempfile
import os


# -----------------------------
# Configuration
# -----------------------------

LINE_Y_TOL = 2.0
MIN_COLUMN_GAP = 10.0

GUTTER_SEARCH_LO = 0.35
GUTTER_SEARCH_HI = 0.65

CROSS_MARGIN = 2.0


# -----------------------------
# Run pdftotext -bbox
# -----------------------------

def run_pdftotext_bbox(pdf_path):
    with tempfile.TemporaryDirectory() as td:

        out_xml = os.path.join(td, "out.xml")

        subprocess.run(
            ["pdftotext", "-bbox", pdf_path, out_xml],
            check=True,
            capture_output=True
        )

        with open(out_xml, "r", encoding="utf-8") as f:
            return f.read()


# -----------------------------
# Parse PDF pages
# -----------------------------

def parse_pages(xml_text):

    root = ET.fromstring(xml_text)

    pages = []

    for page_el in root.iter():

        if page_el.tag.endswith("page"):

            width = float(page_el.attrib["width"])
            height = float(page_el.attrib["height"])

            words = []

            for w in page_el:

                if w.tag.endswith("word"):

                    text = (w.text or "").strip()

                    if not text:
                        continue

                    words.append({
                        "x0": float(w.attrib["xMin"]),
                        "x1": float(w.attrib["xMax"]),
                        "y0": float(w.attrib["yMin"]),
                        "y1": float(w.attrib["yMax"]),
                        "text": text,
                    })

            pages.append({
                "width": width,
                "height": height,
                "words": words
            })

    return pages


# -----------------------------
# Group words into lines
# -----------------------------

def group_lines(words):

    if not words:
        return []

    ws = sorted(
        words,
        key=lambda w: (w["y0"], w["x0"])
    )

    lines = []

    cur = [ws[0]]
    cur_y = ws[0]["y0"]

    for w in ws[1:]:

        if abs(w["y0"] - cur_y) <= LINE_Y_TOL:

            cur.append(w)

        else:

            lines.append(cur)

            cur = [w]
            cur_y = w["y0"]

    lines.append(cur)

    # Sort each line left → right
    for line in lines:
        line.sort(key=lambda w: w["x0"])

    return lines


# -----------------------------
# Find candidate gutter
# -----------------------------

def find_line_gap(line, width):

    lo = width * GUTTER_SEARCH_LO
    hi = width * GUTTER_SEARCH_HI

    best_gap = 0.0
    best = None

    for i in range(len(line) - 1):

        gap = line[i + 1]["x0"] - line[i]["x1"]

        mid = (
            line[i + 1]["x0"] +
            line[i]["x1"]
        ) / 2.0

        if gap > best_gap and lo <= mid <= hi:

            best_gap = gap
            best = (
                line[i]["x1"],
                line[i + 1]["x0"]
            )

    if best is not None and best_gap >= MIN_COLUMN_GAP:
        return best

    return None


# -----------------------------
# Estimate page gutter
# -----------------------------

def estimate_page_gutter(lines, width):

    samples = [
        find_line_gap(line, width)
        for line in lines
    ]

    samples = [
        s for s in samples
        if s is not None
    ]

    # Not enough evidence of columns
    if len(samples) < 3:
        return None

    los = sorted(s[0] for s in samples)
    his = sorted(s[1] for s in samples)

    return (
        los[len(los) // 2],
        his[len(his) // 2]
    )


# -----------------------------
# Classify a line
# -----------------------------

def classify_line(line, gutter):

    g_lo, g_hi = gutter

    def overlap(w):

        return (
            min(w["x1"], g_hi)
            - max(w["x0"], g_lo)
        )

    crossing = [
        w for w in line
        if overlap(w) > CROSS_MARGIN
    ]

    # Word actually crosses the gutter
    if crossing:
        return ("span", line)

    left = [
        w for w in line
        if w["x1"] <= g_lo + CROSS_MARGIN
    ]

    right = [
        w for w in line
        if w["x0"] >= g_hi - CROSS_MARGIN
    ]

    if left and right:
        return ("split", left, right)

    if left:
        return ("left", left)

    if right:
        return ("right", right)

    return ("span", line)


# -----------------------------
# Convert words → text
# -----------------------------

def line_text(line):

    return " ".join(
        w["text"]
        for w in line
    )


# -----------------------------
# Render one page
# -----------------------------

def render_page(page, debug=False):

    lines = group_lines(page["words"])

    if not lines:
        return ""

    gutter = estimate_page_gutter(
        lines,
        page["width"]
    )

    if debug:
        print(
            f"Gutter estimate: {gutter}"
        )

    out_chunks = []

    # Single-column page
    if gutter is None:

        for line in lines:
            out_chunks.append(
                line_text(line)
            )

        return "\n".join(out_chunks)

    left_buf = []
    right_buf = []

    def flush():

        nonlocal left_buf, right_buf

        for line in left_buf:
            out_chunks.append(
                line_text(line)
            )

        if left_buf and right_buf:
            out_chunks.append("")

        for line in right_buf:
            out_chunks.append(
                line_text(line)
            )

        left_buf = []
        right_buf = []

    # Process lines
    for line in lines:

        kind, *parts = classify_line(
            line,
            gutter
        )

        if kind == "left":

            left_buf.append(parts[0])

        elif kind == "right":

            right_buf.append(parts[0])

        elif kind == "split":

            left_buf.append(parts[0])
            right_buf.append(parts[1])

        else:
            # Full-width line
            flush()

            out_chunks.append(
                line_text(parts[0])
            )

    flush()

    return "\n".join(out_chunks)


# -----------------------------
# Extract entire PDF
# -----------------------------

def extract_pdf_text(pdf_path, debug=False):

    xml_text = run_pdftotext_bbox(pdf_path)

    pages = parse_pages(xml_text)

    all_text = []

    for i, page in enumerate(pages):

        if debug:
            print(f"Processing page {i + 1}")

        page_text = render_page(
            page,
            debug=debug
        )

        all_text.append(
            f"--- page {i + 1} ---\n"
            + page_text
        )

    return "\n\n".join(all_text)

In [6]:
text = extract_pdf_text(pdf_path)

FileNotFoundError: [WinError 2] The system cannot find the file specified